In [ ]:
from pyspark.sql.functions import col, avg, sum as spark_sum, unix_timestamp, to_timestamp

In [ ]:
attendance_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/data/attendance.csv")
tasks_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/data/tasks.csv")

In [ ]:
attendance_df = attendance_df.dropna(subset=["employeeid", "clockin", "clockout"])
attendance_df = attendance_df.withColumn("clockin", to_timestamp("clockin"))
attendance_df = attendance_df.withColumn("clockout", to_timestamp("clockout"))
attendance_df = attendance_df.withColumn("workhours", (unix_timestamp("clockout") - unix_timestamp("clockin")) / 3600)

In [ ]:
combined_df = attendance_df.join(tasks_df, on=["employeeid", "department"], how="left")

In [ ]:
department_kpis = combined_df.groupBy("department").agg(
    avg("workhours").alias("avg_workhours"),
    spark_sum("taskscompleted").alias("total_tasks_completed"),
    avg("productivityscore").alias("avg_productivity_score")
)
department_kpis.show()

In [ ]:
department_kpis.write.format("delta").mode("overwrite").save("/mnt/data/department_kpis_delta")
department_kpis.write.format("csv").option("header", "true").mode("overwrite").save("/mnt/data/department_kpis_csv")